# .CZI to .TIFF Conversion Notebook

### Setup
Welcome to this Jupyter Notebook for converting Ziess .CZI images into TIFF images. It is designed to operate on images with BGR48 pixel types, with all information in a single 2d layer, and export to a 16 bit .tiff file with dynamic range scaling. Other formats will require modification to the image processing. 

If you haven't used a Jupyter notebook before, it is fairly simple. First, you must have a "kernel" selected. This is likely an option in the top right corner, or somewhere else in the menus. Setting it up is fairly straightforward, but is probably better explained by a web search or chatbot prompt. It is worth noting that this script **presently only works with python 3.13.x,** and thus the kernel must be of this version of python. In order to use this program, you must run various cells in the notebook. A cell is a box of code or text, such as this one, or the code box below. Code boxes can be run using the play button on their left side. When a code box is run, its output will be shown below the cell. Cells should generally be run sequentially unless otherwise specified. 

This next cell installs the python libraries and dependencies that need to be run. It must be run once before any other cells can be. 

In [ ]:
! pip install --upgrade pip
! pip install jupyter --upgrade
! pip install ipywidgets --upgrade
! pip install pylibCZIrw matplotlib tifffile

The next cell imports the required python libraries in order to do any conversion. It also must be run once before any other cells can be. 

In [ ]:
from pylibCZIrw import czi as pyczi
import tifffile
import matplotlib.pyplot as plt
import numpy as np
import tkinter as tk
from tkinter import filedialog, simpledialog, messagebox
import os
import re
from multiprocessing import Pool

### Opening Files

Next, we need to link the conversion program to the files that we want to convert. You can specify a single file for conversion, or a directory (folder) containing all the desired files.

**1.** Select processing mode:
> **A** Directory mode (batch export, no preview). 
> > -  Step 1: Select the **input directory (folder).** This should be the folder where all the .CZI files you want to convert are located. File names do not matter. It will convert (or attempt to convert) any and all .CZI files present in this directory, but will not check recursively (that is it will not check subdirectories within this directory)
> > -  Step 2: Select the **output directory**. These can be the same directory, which will simply place the converted files in the existing directory, without removing any other existing files. It can also be any other directory, in which case it will place the files in that directory. It cannot create a new directory, you must select one that exists.
> > -  (Optional) Step 3: Specify **output file names.**  The name provided will be prefixed to the index of the file when exporting. Files will be exported in sequential order. That is, the first file loaded will be exported as "{specified name}1.tiff" **Please note, if files with the same name and extension (other tiff files) exist in the output directory, they WILL be OVERWRITTEN** Please use unique names, or specify a different output directory. **Please specify just the prefix, not including the index or .tiff extension.**
> > - (Optional) Step 5: specify start index . If the desired starting index is not 0, enter the desired value here. Close/cancel window to use default (0)

> **B** Single Output Mode (One image, with preview)
> > -  Step 1: Select the **input file.** This should be the .CZI you want to convert
> > -  Step 2: Specify the **output file.** This file should not yet exist. If it does, it will prompt you if you would like to overwrite it. If you select yes, it will overwrite the existing file. 

In either case, the output should state the current mode, and the input and output paths, as well as the output name

In [ ]:
root = tk.Tk()
root.withdraw()
root.call('wm', 'attributes', '.', '-topmost', True)

choice = tk.StringVar(value="")

dialog = tk.Toplevel(root)
dialog.title("Export format")
dialog.resizable(False, False)
dialog.lift()
dialog.focus_force()

tk.Label(dialog, text="Select Operation Mode:", pady=10).pack()

btn_frame = tk.Frame(dialog)
btn_frame.pack(pady=10, padx=20)

tk.Button(btn_frame, text="Directory Batch Mode", width=24,
          command=lambda: (choice.set("Directory Batch Mode"), dialog.destroy())).pack(side=tk.LEFT, padx=5)
tk.Button(btn_frame, text="Single File Mode", width=24,
          command=lambda: (choice.set("Single File Mode"), dialog.destroy())).pack(side=tk.LEFT, padx=5)

dialog.wait_window()
root.destroy()

if not choice.get():
    raise SystemExit("No option selected — re-run this cell to try again.")

if choice.get() == "Directory Batch Mode":
    batchMode = True
elif choice.get() == "Single File Mode":
    batchMode = False
else:
    raise SystemExit("Invalid Operation Mode")
print(f"{choice.get()} Selected")

if batchMode:
    root = tk.Tk()
    root.withdraw()
    root.call('wm', 'attributes', '.', '-topmost', True)

    inputDirectory = filedialog.askdirectory(
        title="Select input folder"
    )

    

    if not inputDirectory:
        raise SystemExit("No folder selected — re-run this cell to try again.")

    print(f"Selected Input Directory: {inputDirectory}")
    
    root.withdraw()
    root.call('wm', 'attributes', '.', '-topmost', True)

    outputDirectory = filedialog.askdirectory(
        title="Select output folder"
    )

    if not outputDirectory:
        raise SystemExit("No folder selected — re-run this cell to try again.")

    print(f"Selected Output Directory: {outputDirectory}")
    
    root.withdraw()
    root.call('wm', 'attributes', '.', '-topmost', True)

    result = messagebox.askquestion(
        title="Use Original Filenames",
        message="Use Original Filenames?\n\nYes -> Use original filenames \nNo -> Specify new filename and index"
    )

    if result == "no":
        root.withdraw()
        root.call('wm', 'attributes', '.', '-topmost', True)

        outputFilename = filedialog.asksaveasfilename(
            title="Save TIFF as: (JUST ENTER THE PREFIX)",
            defaultextension=".tiff",
            filetypes=[("TIFF files", "*.tiff"), ("All files", "*.*")],
            initialdir=outputDirectory
        )

        if outputFilename:
            outputFilename = outputFilename.split(os.sep)[-1].removesuffix(".tiff")
            print(f"Output Filename: {outputFilename}[index].tiff")
        else:
            outputFilename = None
            print("No output filename specified. Using original filenames.")
            
    else: 
        outputFilename = None
        print(f"Using Original Filenames")
    
    if outputFilename != None:
        root.withdraw()  # Hide the main window

        initialIndex = simpledialog.askfloat("Input", "Enter Starting Index:")
        initialIndex = 0 if initialIndex is None else int(initialIndex)
        print(f"Starting Index: {initialIndex}")
    
    root.destroy()
    
else:
    root = tk.Tk()
    root.withdraw()  # Hide the empty root window
    root.call('wm', 'attributes', '.', '-topmost', True)  # Bring dialog to front

    inputFilepath = filedialog.askopenfilename(
        title="Select a CZI file",
        filetypes=[("CZI files", "*.czi"), ("All files", "*.*")]
    )

    if not inputFilepath:
        raise SystemExit("No file selected — re-run this cell to try again.")

    print(f"Selected Input File: {inputFilepath}")
    
    # Ask where to save
    root.withdraw()
    root.call('wm', 'attributes', '.', '-topmost', True)

    save_path = filedialog.asksaveasfilename(
        title="Save TIFF as",
        defaultextension=".tiff",
        filetypes=[("TIFF files", "*.tiff"), ("All files", "*.*")]
    )

    root.destroy()

    if not save_path:
        raise SystemExit("No save location selected — re-run this cell to try again.")
    
    print(f"Selected Save Location: {save_path}")
    outputFilename = save_path.split("/")[-1]
    print(f"Output Filename: {outputFilename}")
    

**Please verify that the above information appears correct.** Note the directory links may be broken if your folder tree contains folders with spaces in the name, but it will otherwise function. If it is not correct, re-run the above cell to correct it.

This next cell is here so that you can verify that the files being loaded are the correct type for this script. For batch operations, it will print all unique formats. This takes significant time in batch mode, as it must load every file to check it. As such, **skip this cell if you already know that the images are in the correct format** to save time.

The output for each format should be:

```
Dimensions: {'T': (0, 1), 'Z': (0, 1), 'C': (0, 1), 'H': (0, 1), 'X': (0, {Image Width}), 'Y': (0, {Image Height})}
Plane shape: ({Image Height}, {Image Width}, 3)
{0: 'Bgr48'}
```

In order for this script to work, the last line must have Bgr48 in it, and the 2nd line should have a 3 for the third value.

(You may have to change the output to a scrollable element if there are many different formats.)

In [ ]:
def check_image_format(inputFilepath):
    try:
        with pyczi.open_czi(inputFilepath) as czidoc:
            bbox = czidoc.total_bounding_box
            img = czidoc.read(plane={"T": 0, "Z": 0, "C": 0}, zoom=1.0)
            pixtype = czidoc.pixel_types
            return(f"""Dimensions: {bbox},
                   Plane shape: {img.shape},
                   Pixel type: {pixtype}""")
    except Exception as e:
        print(f"Error occurred while checking {inputFilepath}: {e}")
        return None

if batchMode:
    formats = set([check_image_format(os.path.join(inputDirectory, f)) for f in os.listdir(inputDirectory) if (f.endswith(".czi") or f.endswith(".CZI")) and not f.startswith(".")])
    for format in formats: print(format)
else:
    print(check_image_format(inputFilepath))

The next cell generates a low resolution preview that can be viewed in the notebook output using matplotlib. It will not have the full resolution of the original or final exported images, but all other aspects should be the same. 

This preview will only be shown in Single File Output mode

In [ ]:
if not batchMode:
    with pyczi.open_czi(inputFilepath) as czidoc:
        img = czidoc.read(plane={"T": 0, "Z": 0, "C": 0}, zoom=0.25)

    img_rgb = img[:, :, ::-1].astype(np.float32) / 65535.0

    p_low, p_high = np.percentile(img_rgb, (1, 99))
    img_stretched = np.clip((img_rgb - p_low) / (p_high - p_low), 0, 1)

    plt.figure(figsize=(10, 12))
    plt.imshow(img_stretched)
    plt.axis("off")
    plt.show()
else:
    print("Batch mode is selected. Image preview is not available. Please continue to the next cell, or return to the beginning and select single file mode.")

The last cell here is where the actual file conversion happens. The process is safe for the original files, as they are never modified or overwritten. However, please verify that the specified output filename will not overwrite any desired files in the output directory (ensure no files share the same naming scheme, unless you want them to be overwritten).

Once the cell is run, it will begin converting files automatically. The output will log each sucessfull file export's name and location, as well as any errors it encounters. 

Note: This export process does technically alter the image information, as it streches the pixel values to fill the full dynamic range. This ensures that the image doesn't just appear "black" as a result of relatively low value usage of the available dynamic range (for example, having white pixels with RGB values of (4000,4000,4000) should be converted to (65535, 65535, 65535)). This shouldn't alter the relative content of the image, and maintains 16 bit precision as in the original image. If you do not want this streching (such as if you want to process the images some other way afterwards, and would like just the raw pixel data), change the STRECH_DYNAMIC_RANGE flag to False.

Note 2: Running this cell more than once will overwrite previous runs, as it simply writes files to the same directory and names as it originally did. 

Note 3: This script defaults to not being multithreaded. This is to try and prevent memory shortages with very large files. Change the MULTITHREADED flag to True to allow multithreading, and set the maxiumum number of threads with the MAX_THREADS constant (will automatically limit to the number of cores available)

In [ ]:
STRECH_DYNAMIC_RANGE = True
MULTITHREADED = True
MAX_THREADS = 6

def exportFullSizeTIFF(inputFilepath, save_path, img_name):
    try:
        with pyczi.open_czi(inputFilepath) as czidoc:
            img_full = czidoc.read(plane={"T": 0, "Z": 0, "C": 0}, zoom=1.0)

        if STRECH_DYNAMIC_RANGE:
            img_full_rgb = img_full[:, :, ::-1].astype(np.float32)                      # Swap BGR -> RGB, convert to 32 bit floating point values for accurate scaling
            p_low, p_high = np.percentile(img_full_rgb, (1, 99))                        # Calculate percentiles for dynamic range expansion
            img_stretched = np.clip((img_full_rgb - p_low) / (p_high - p_low), 0, 1)    # Rescale existing pixel values using percentiles
            img_16bit = (img_stretched * 65535).astype(np.uint16)                       # Convert floating point percentage values for each pixel back down to a 16 bit integer
        else:
            img_16bit = img_full[:, :, ::-1]                                            # Swap BGR -> RGB

        tifffile.imwrite(save_path, img_16bit)

        print(f"Saved {img_name} to: {save_path}")
    except Exception as e: print(f"Error occurred while processing {inputFilepath}: {e}")

def natural_sort_key(filename):
    # Split filename into text and number chunks
    parts = re.split(r'(\d+)', filename)
    return [int(part) if part.isdigit() else part.lower() for part in parts]
    
if batchMode:
    inputfiles = sorted([f for f in os.listdir(inputDirectory) if (f.endswith(".czi") or f.endswith(".CZI")) and not f.startswith(".")], key=natural_sort_key)
    processingFilepaths = []
    for index, filename in enumerate(inputfiles):
        inputFilepath = os.path.join(inputDirectory, filename)
        save_path = os.path.join(outputDirectory, f"{outputFilename}{initialIndex+index}.tiff" if outputFilename else f"{filename.removesuffix('.czi').removesuffix('.CZI')}.tiff")
        processingFilepaths.append((inputFilepath, save_path, filename))
          
    if __name__ == '__main__':
        cpu_count = os.cpu_count() if MULTITHREADED else 1
        cpu_count = min(MAX_THREADS, cpu_count) if cpu_count is not None else 1
        if cpu_count > 1: print(f"Using {cpu_count} threads. BEWARE OF MEMORY USAGE")
        with Pool(cpu_count) as pool:
            results = pool.starmap(exportFullSizeTIFF, processingFilepaths)
        
else:
    exportFullSizeTIFF(inputFilepath, save_path, inputFilepath.split(os.sep)[-1])